## Count data of deprecated and new dataset versions

## Final test: `dataset_new`

This test checks for duplicate image content and counts images for every class in each dataset.

In [ ]:
import hashlib
from collections import defaultdict
from pathlib import Path


new_dataset_root = Path("dataset_new")
if not new_dataset_root.is_dir():
    new_dataset_root = Path("../dataset_new")

class_names = {0: "Cross", 1: "Circle", 2: "Triangle", 3: "Rectangle"}
per_dataset_counts = {}
per_class_counts = defaultdict(int)
seen_images = defaultdict(list)

for dataset_dir in sorted(new_dataset_root.iterdir(), key=lambda path: path.name):
    if not dataset_dir.is_dir():
        continue

    dataset_counts = {label: 0 for label in class_names}
    for class_dir in sorted(dataset_dir.iterdir(), key=lambda path: path.name):
        if not class_dir.is_dir():
            continue
        try:
            label = int(class_dir.name)
        except ValueError:
            continue
        if label not in class_names:
            raise AssertionError(f"Invalid class folder: {class_dir}")

        for image_path in sorted(class_dir.iterdir(), key=lambda path: path.name):
            if not image_path.is_file():
                continue
            digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
            seen_images[digest].append(image_path)
            dataset_counts[label] += 1
            per_class_counts[label] += 1

    per_dataset_counts[dataset_dir.name] = dataset_counts

duplicate_groups = {
    digest: paths for digest, paths in seen_images.items() if len(paths) > 1
}
duplicate_count = sum(len(paths) - 1 for paths in duplicate_groups.values())

print("| Dataset | Cross | Circle | Triangle | Rectangle | Number of images |")
print("|---|---:|---:|---:|---:|---:|")
for dataset_name, counts in per_dataset_counts.items():
    row = [counts[label] for label in class_names]
    print(f"| {dataset_name} | " + " | ".join(map(str, row)) + f" | {sum(row)} |")

full_row = [per_class_counts[label] for label in class_names]
full_total = sum(full_row)
print(f"| **dataset_new** | " + " | ".join(map(str, full_row)) + f" | **{full_total}** |")
print(f"\nDuplicate image content groups: {len(duplicate_groups)}")
print(f"Duplicate files: {duplicate_count}")

assert duplicate_count == 0, duplicate_groups
assert full_total == sum(sum(counts.values()) for counts in per_dataset_counts.values())

In [ ]:
import os

# Folder structure of deprecated datasets is: 
# - dataset1
#   - data-folder with (RGB) images
#   - gt-folder with .txt-File for every corresponding image in folder data with class label (e.g. 0 = cross, 1 = circle, 2 = triangle and 3 = rectangle)
# - dataset 1
# - ...
# - dataset 9
# - 0 = cross
# - 1 = circle
# - 2 = triangle
# - 3 = rectangle

#- goal: count and print the respective numbers for all 4 classes in a table for all 9 datasets
#- goal 2: output image dimensions for each dataset (width, height, channels) and 
# print them in a table for all 9 datasets, but only once for different image dimensions 
# (e.g. if all images in dataset1 have the same dimensions, only print them once)
folders = ["dataset1", "dataset2", "dataset3", "dataset4", "dataset5", "dataset6", "dataset7", "dataset8", "dataset9", "dataset11"]

for folder in folders:
    # count the number of images in the data-folder
    data_folder = folder + "/data"
    gt_folder = folder + "/gt"
    num_images = len([name for name in os.listdir(data_folder) if os.path.isfile(os.path.join(data_folder, name))])
    
    # count the number of class labels in the gt-folder
    class_counts = [0, 0, 0, 0]  # [cross, circle, triangle, rectangle]
    for filename in os.listdir(gt_folder):
        if filename.endswith(".txt"):
            with open(os.path.join(gt_folder, filename), 'r') as f:
                label = int(f.read().strip())
                class_counts[label] += 1
    
    print(f"Dataset: {folder}")
    print(f"Number of images: {num_images}")
    print(f"Class counts: Cross: {class_counts[0]}, Circle: {class_counts[1]}, Triangle: {class_counts[2]}, Rectangle: {class_counts[3]}")
    print("-" * 40)

    print(f"Image dimensions for {folder}:")
    # Get image dimensions
    image_dimensions = set()
    for filename in os.listdir(data_folder):
        if filename.endswith(".jpg") or filename.endswith(".png"):
            from PIL import Image
            with Image.open(os.path.join(data_folder, filename)) as img:
                image_dimensions.add(img.size + (len(img.getbands()),))  # (width, height, channels)

    for width, height, channels in image_dimensions:
        print(f"  Width: {width}, Height: {height}, Channels: {channels}")
    

### Image count:
| Dataset | Cross | Circle | Triangle | Rectangle | Number of images | Image dimension* |
|---|---:|---:|---:|---:|---:|---|
| dataset1 | 44 | 45 | 44 | 43 | 176 | 640 × 480 | 
| dataset2 | 37 | 35 | 36 | 36 | 144 | 640 × 480 |
| dataset3 | 30 | 29 | 30 | 31 | 120 | 640 × 480 |
| dataset4 | 86 | 86 | 94 | 91 | 357 | 640 × 480 |
| dataset5 | 175 | 175 | 174 | 174 | 698 | 638@1920×1080 + 60@640×480 |
| dataset6 | 60 | 60 | 60 | 60 | 240 | 1920 × 1080 |
| dataset_new | 432 | 430 | 438 | 435 | 1735 | RGB |

*RGB-images

## Arbeitsauftrag 1:
### Deadline: 10. November um 20 Uhr 
#### `create-symbols-dataset.py`

Assuming numbering is sequential, starting at **1400.png**, with each pupil’s 10 images per class grouped together in the order Cross, Circle, Triangle, Rectangle:

| Pupil | Cross | Circle | Triangle | Rectangle |
|---|---|---|---|---|
| 1 | 1400–1409.png | 1410–1419.png | 1420–1429.png | 1430–1439.png |
| 2 | 1440–1449.png | 1450–1459.png | 1460–1469.png | 1470–1479.png |
| 3 | 1480–1489.png | 1490–1499.png | 1500–1509.png | 1510–1519.png |
| 4 | 1520–1529.png | 1530–1539.png | 1540–1549.png | 1550–1559.png |
| 5 | 1560–1569.png | 1570–1579.png | 1580–1589.png | 1590–1599.png |
| 6 | 1600–1609.png | 1610–1619.png | 1620–1629.png | 1630–1639.png |

That assigns **240 images total**, ending at **1639.png**.